In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
import pickle
from datetime import datetime
from pathlib import Path

import sys
sys.path.append('../../')
from codebase.pairwise_copy_spatial_simulations import Simulate_Fish_School, Simulate_Fish_School_Excluded_Volume

from codebase.subgroup_analysis import get_num_neighbours, assign_state_variable_and_get_transitions, _generate_sum_partitions
from codebase.parameter_fitting import fit_s_epsilon_jointly_bounded

In [ ]:
fish_ages = [2,4,6,8] # weeks
bodylengths = [4,7,10,12] # mm, from Zada et. al. 2024
arena_radii = [75, 150, 150, 150] # mm, radius of experimental arena for each age group
L = 50 #mm 
optimal_omega = 0.032 # coefficient of excluded volume interaction turns

microscopic_params_full_dataset = pd.read_csv('../../Results/microscopic_params_full_dataset.csv').to_numpy()
param_distribution_data = np.load('../../Results/parameter_distribution_from_subsamples.npz')

# Run simulations with randomly sampled microscopic parameters

In [20]:
def sample_params_run_sims_and_analyze_results(N_fish=4, 
                                 Tmax=500, 
                                 dt=0.01, 
                                 N_sims=20, 
                                 N_samples=20, 
                                 param_distribution_data=param_distribution_data, 
                                 arena_radii=arena_radii, 
                                 L=L,
                                 excluded_volume=False,
                                 optimal_omega=optimal_omega,
                                 fish_ages=fish_ages, 
                                 random_state=0
                                 ):

    def sample_positive_mvn(mean, cov, N_samples, random_state):
        # Create a random number generator
        rng = np.random.default_rng(random_state)
        samples = []
        while len(samples) < N_samples:
            candidate = rng.multivariate_normal(mean, cov)
            if np.all(candidate > 0):
                samples.append(candidate)
        return np.array(samples)
    
    # enumerate system states i.e. subgroup configurations
    states_list_og = _generate_sum_partitions(N_fish)
    if N_fish == 4:
        #switch order of [2,2] and [1,3]!
        order = [0,1,3,2,4]
        states_list = [states_list_og[i] for i in order]
    else:
        states_list = states_list_og
    Nstates = len(states_list)

    frames = int(Tmax/dt) + 1
    N_ages = len(fish_ages)
    # arrays to save results
    sampled_params = np.empty((N_ages, N_samples, 4)) # s, eps, c, v
    neighbors_all_ages_and_samples = np.empty((N_ages,N_samples,N_sims, N_fish, frames))
    transition_matrices_all_ages_and_samples = np.empty((N_ages,N_samples, N_sims, Nstates, Nstates))
    sys_state_var_all_ages_and_samples = np.empty((N_ages, N_samples, N_sims, frames), dtype=int)

    for j, age in enumerate(fish_ages):

        R = arena_radii[j] #mm, radius of arena for simulation

        # sample parameters
        mean = param_distribution_data['agewise_mean'][j,:]
        cov = param_distribution_data['agewise_cov'][j,:,:]
        s_upper_bound = param_distribution_data['s_upper_bounds'][j]
        random_sample = sample_positive_mvn(mean, cov, N_samples, random_state)

        # record microscopic parameters for each sample
        sampled_params[j,:,3] = random_sample[:,3] #v
        sampled_params[j,:,2] = random_sample[:,1] #c
        # s,epsilon will be calculated below from alpha and slope for each sample

        for i in range(N_samples):
            # calculate s, epsilon for the sample
            alpha = random_sample[i,0]
            slope = random_sample[i,2]
            s, epsilon = fit_s_epsilon_jointly_bounded(alpha, slope, s_upper_bound)
            sampled_params[j,i,0] = s
            sampled_params[j,i,1] = epsilon

            for kk in range(N_sims):
                s = np.ones(N_fish)*sampled_params[j,i,0] # s^-1, rate of diffusion
                eps = np.ones(N_fish)*sampled_params[j,i,1] # variance of diffusion noise
                c = np.ones(N_fish)*sampled_params[j,i,2] # s^-1, rate of copying 
                v = np.ones(N_fish)*sampled_params[j,i,3] # mm/s, speed

                if excluded_volume:
                    sim = Simulate_Fish_School_Excluded_Volume(N=N_fish, 
                                    v=v, 
                                    R=R, 
                                    s=s, 
                                    eps=eps, 
                                    c=c, 
                                    Tmax=Tmax, 
                                    dt=dt, 
                                    L=L, 
                                    omega=optimal_omega, 
                                    R_ex = bodylengths[j], 
                                    seed=kk)
                else:
                    sim = Simulate_Fish_School(N=N_fish, 
                                    v=v, 
                                    R=R, 
                                    s=s, 
                                    eps=eps, 
                                    c=c, 
                                    Tmax=Tmax, 
                                    dt=dt, 
                                    L=L, 
                                    seed=kk)

                x,y,theta = sim.run_simulation()
                # subgroup analysis
                sys_state_var, transitions = assign_state_variable_and_get_transitions(x,y,L,states_list)
                sys_state_var_all_ages_and_samples[j,i,kk,:] = sys_state_var
                transition_matrices_all_ages_and_samples[j,i,kk,:,:] = transitions
                # interacting neighbors
                neighbors_all_ages_and_samples[j,i,kk,:,:] = get_num_neighbours(x,y,L).reshape(N_fish,-1)

            if i%10==0:
                print(f"Sample {i+1}/{N_samples} for {age}wpf completed at {datetime.now().strftime('%H:%M:%S')}")

    # save analysis results
    analysis_results_sims = {
        'sampled_parameters': sampled_params,
        'neighbours': neighbors_all_ages_and_samples,
        'sys_state_var': sys_state_var_all_ages_and_samples,
        'transition_matrix': transition_matrices_all_ages_and_samples,
        'parameter_names' : param_distribution_data['parameter_names'],
        }
        
    return analysis_results_sims


In [37]:
N_samples = 20
N_sims = 20
Tmax = 600
dt = 0.01

### Without excluded volume

In [38]:
results_path = Path('../../Results/model_sims_no_exclusion_results.pkl')
if results_path.is_file():
    with open(results_path, 'rb') as f:
        model_sims_no_exclusion_results = pickle.load(f)
else:
    model_sims_no_exclusion_results = sample_params_run_sims_and_analyze_results(N_fish=4,
                                                                           Tmax=Tmax,
                                                                            dt=dt,
                                                                            N_sims=N_sims,
                                                                            N_samples=N_samples,
                                                                            excluded_volume=False)

    # save results
    with open(results_path, 'wb') as f:
        pickle.dump(model_sims_no_exclusion_results, f)


Sample 1/20 for 2wpf completed at 22:13:06
Sample 11/20 for 2wpf completed at 22:21:19
Sample 1/20 for 4wpf completed at 22:28:45
Sample 11/20 for 4wpf completed at 22:36:19
Sample 1/20 for 6wpf completed at 22:44:09
Sample 11/20 for 6wpf completed at 22:52:13
Sample 1/20 for 8wpf completed at 23:00:23
Sample 11/20 for 8wpf completed at 23:08:08
